Step 3: Model and prediction

Fits a Quantile Regression Forest (QRF) for each soil property (pH and SOC, 0–30 cm), using the point table from step 1 and the covariate stack from step 2.

Covariate extraction: covariate values are sampled at each site. Sites on no-data pixels are dropped.
Spatial cross-validation: sites are grouped into 10 km blocks, so nearby sites are never split between training and testing. Random cross-validation is run alongside it for comparison. Metrics: bias (ME), RMSE, MAE, R², concordance (CCC) and PICP90, the share of observations inside the 90% prediction interval.
Final model: trained on all sites, with covariate importance recorded.
Area of Applicability (AOA): the dissimilarity index (DI) measures how different each pixel's covariates are from the training data. Pixels above the threshold are flagged as outside the AOA.
Prediction: every pixel gets a median prediction and a 90% prediction interval.

Mount drive and allow access

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip install quantile-forest rasterio pyproj scikit-learn scipy pandas matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 30.3 MB/s eta 0:00:00


Set up project folder and copy data

In [6]:
import os, shutil, glob

PROJ = '/content/drive/MyDrive/soil_dss'            # project folder
GEE_FOLDER = '/content/drive/MyDrive/soil_dss_covariates'  # GEE export folder

os.makedirs(f'{PROJ}/data/covariates', exist_ok=True)
for f in glob.glob(f'{GEE_FOLDER}/cov_kak_tnz_100m*'):
    shutil.copy(f, f'{PROJ}/data/covariates/')

os.chdir(PROJ)
print('data:', os.listdir('data'))
print('covariates:', os.listdir('data/covariates'))

data: ['covariates', 'points_ph_soc_kakamega_transnzoia.csv']
covariates: ['cov_kak_tnz_100m_bands.csv', 'cov_kak_tnz_100m.tif']


Import libraries

In [13]:
import glob
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from pyproj import Transformer
from quantile_forest import RandomForestQuantileRegressor
from rasterio.merge import merge
from rasterio.windows import Window
from scipy.spatial import cKDTree
from scipy.spatial.distance import pdist
from sklearn.model_selection import GroupKFold, KFold

Read data and set parameters

In [14]:
POINTS = "data/points_ph_soc_kakamega_transnzoia.csv"
COV_PATTERN = "data/covariates/cov_kak_tnz_100m*.tif"
COV_MERGED = "data/covariates/cov_kak_tnz_100m_merged.tif"
BANDS_CSV = "data/covariates/cov_kak_tnz_100m_bands.csv"
PROPS = {"ph": "pH", "soc_gkg": "SOC (g/kg)"}
NODATA = -9999

BLOCK_M = 10_000       # spatial CV block size; larger than the Trans Nzoia clusters
N_FOLDS = 5
QUANTILES = [0.05, 0.25, 0.5, 0.75, 0.95]
QNAMES = ["q05", "q25", "q50", "q75", "q95"]
BAND_NAMES = QNAMES + ["pi90_width", "di", "aoa"]
NB = len(BAND_NAMES)
RF = dict(n_estimators=500, min_samples_leaf=3, max_features=0.33,
          random_state=42, n_jobs=-1)
ROW_CHUNK = 128        # raster rows predicted at a time (lower if memory is tight)

OUT_M, OUT_MAPS = Path("outputs/model"), Path("outputs/maps")
OUT_M.mkdir(parents=True, exist_ok=True)
OUT_MAPS.mkdir(parents=True, exist_ok=True)

Covariates

In [15]:
# ------------------------------------------------------------------ covariates
def covariate_raster():
    """Return the covariate GeoTIFF path, merging GEE export tiles if needed."""
    tiles = sorted(p for p in glob.glob(COV_PATTERN) if "merged" not in p)
    if not tiles:
        raise FileNotFoundError(f"No covariate GeoTIFF matching {COV_PATTERN}")
    if len(tiles) == 1:
        return tiles[0]
    if not Path(COV_MERGED).exists():
        print(f"Merging {len(tiles)} export tiles ...")
        srcs = [rasterio.open(t) for t in tiles]
        mosaic, transform = merge(srcs, nodata=NODATA)
        prof = srcs[0].profile
        prof.update(height=mosaic.shape[1], width=mosaic.shape[2],
                    transform=transform, nodata=NODATA, compress="deflate",
                    tiled=True, BIGTIFF="IF_SAFER")
        with rasterio.open(COV_MERGED, "w", **prof) as dst:
            dst.write(mosaic)
        for s in srcs:
            s.close()
    return COV_MERGED


COV = covariate_raster()
band_names = (pd.read_csv(BANDS_CSV).sort_values("band_index")["band_name"].tolist())

sites = pd.read_csv(POINTS)
with rasterio.open(COV) as src:
    assert src.count == len(band_names), (
        f"Raster has {src.count} bands but band list has {len(band_names)}")
    to_rast = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
    xs, ys = to_rast.transform(sites.lon.values, sites.lat.values)
    X_all = np.array(list(src.sample(zip(xs, ys))), dtype="float64")
    profile = src.profile
    raster_crs = src.crs

valid = ~(X_all == NODATA).any(axis=1) & np.isfinite(X_all).all(axis=1)
print(f"Sites: {len(sites)}  with valid covariates: {valid.sum()}")
sites = sites[valid].reset_index(drop=True)
X_all = X_all[valid]
sites["x"], sites["y"] = xs[valid], ys[valid]

# spatial blocks: sites in the same block are never split between train and test
blk = (np.floor(sites.x / BLOCK_M).astype(int).astype(str) + "_" +
       np.floor(sites.y / BLOCK_M).astype(int).astype(str))
sites["block"] = pd.factorize(blk)[0]
print(f"Spatial blocks ({BLOCK_M/1000:.0f} km): {sites.block.nunique()}")

Sites: 509  with valid covariates: 500
Spatial blocks (10 km): 45


Helpers

In [16]:
# ------------------------------------------------------------------ helpers
def qrf():
    return RandomForestQuantileRegressor(**RF)


def metrics(obs, q):
    lo, q25, med, q75, hi = (q[:, i] for i in range(5))
    err = med - obs
    r = np.corrcoef(obs, med)[0, 1]
    ccc = (2 * r * obs.std() * med.std() /
           (obs.var() + med.var() + (obs.mean() - med.mean()) ** 2))
    return {"n": len(obs), "ME": err.mean(), "RMSE": np.sqrt((err ** 2).mean()),
            "MAE": np.abs(err).mean(),
            "R2_MEC": 1 - (err ** 2).sum() / ((obs - obs.mean()) ** 2).sum(),
            "CCC": ccc, "PICP90": ((obs >= lo) & (obs <= hi)).mean(),
            "PICP50": ((obs >= q25) & (obs <= q75)).mean()}


def cross_validate(X, y, splitter, groups=None):
    oof = np.full((len(y), len(QUANTILES)), np.nan)
    for tr, te in splitter.split(X, y, groups):
        oof[te] = qrf().fit(X[tr], y[tr]).predict(X[te], quantiles=QUANTILES)
    return oof


class AOA:
    """Area of Applicability (Meyer & Pebesma 2021): dissimilarity index (DI)
    in importance-weighted, standardised covariate space."""

    def __init__(self, X, importance, folds):
        self.mu, self.sd = X.mean(0), X.std(0)
        self.sd[self.sd == 0] = 1
        self.w = importance / importance.sum()
        Z = self._scale(X)
        self.mean_dist = pdist(Z).mean()
        self.tree = cKDTree(Z)
        # training DI: distance to nearest training point in *other* CV folds
        di_train = np.empty(len(Z))
        for f in np.unique(folds):
            inf, outf = folds == f, folds != f
            d, _ = cKDTree(Z[outf]).query(Z[inf])
            di_train[inf] = d / self.mean_dist
        q1, q3 = np.percentile(di_train, [25, 75])
        self.threshold = min(q3 + 1.5 * (q3 - q1), di_train.max())

    def _scale(self, X):
        return (X - self.mu) / self.sd * self.w

    def di(self, X):
        d, _ = self.tree.query(self._scale(X))
        return d / self.mean_dist

Modelling

In [17]:
# ------------------------------------------------------------------ modelling
summary = []
for prop, label in PROPS.items():
    d = sites.dropna(subset=[prop])
    X, y = X_all[d.index], d[prop].values
    print(f"\n=== {label}: n = {len(y)}")

    # spatial CV (honest) vs random CV (optimistic) - the gap shows clustering bias
    gkf = GroupKFold(n_splits=N_FOLDS)
    oof_sp = cross_validate(X, y, gkf, d.block.values)
    oof_rn = cross_validate(X, y, KFold(N_FOLDS, shuffle=True, random_state=42))
    for name, oof in [("spatial", oof_sp), ("random", oof_rn)]:
        m = {"property": prop, "cv": name, **metrics(y, oof)}
        summary.append(m)
        print(f"  {name:8s} " + "  ".join(f"{k}={v:.3f}" for k, v in m.items()
                                           if isinstance(v, float)))

    out = d[["site_id", "county", "lon", "lat", prop]].copy()
    out[QNAMES] = oof_sp
    out.to_csv(OUT_M / f"{prop}_cv_predictions.csv", index=False)

    # observed vs predicted plot (spatial CV)
    fig, ax = plt.subplots(figsize=(5.5, 5.5))
    for county, g in out.groupby("county"):
        ax.scatter(g[prop], g.q50, s=12, alpha=0.7, label=county)
    lims = [min(y.min(), oof_sp[:, 2].min()), max(y.max(), oof_sp[:, 2].max())]
    ax.plot(lims, lims, "k--", lw=1)
    ms = [s for s in summary if s["property"] == prop and s["cv"] == "spatial"][0]
    ax.set_title(f"{label} - spatial CV\nR² {ms['R2_MEC']:.2f}  RMSE {ms['RMSE']:.2f}  "
                 f"PICP90 {ms['PICP90']:.2f}")
    ax.set_xlabel("Observed"); ax.set_ylabel("Predicted (median)"); ax.legend()
    fig.savefig(OUT_M / f"{prop}_cv_plot.png", dpi=150, bbox_inches="tight")
    plt.close(fig)

    # final model on all data
    model = qrf().fit(X, y)
    imp = pd.Series(model.feature_importances_, index=band_names).sort_values(ascending=False)
    imp.to_csv(OUT_M / f"{prop}_importance.csv", header=["importance"])
    print("  top covariates:", ", ".join(imp.index[:5]))

    folds = np.empty(len(y), dtype=int)
    for i, (_, te) in enumerate(gkf.split(X, y, d.block.values)):
        folds[te] = i
    aoa = AOA(X, model.feature_importances_, folds)

    # ---- predict the raster
    out_path = OUT_MAPS / f"{prop}_0_30cm.tif"
    prof = profile.copy()
    prof.update(count=NB, dtype="float32", nodata=NODATA, compress="deflate",
                tiled=True, blockxsize=256, blockysize=256, BIGTIFF="IF_SAFER")
    with rasterio.open(COV) as src, rasterio.open(out_path, "w", **prof) as dst:
        for row in range(0, src.height, ROW_CHUNK):
            win = Window(0, row, src.width, min(ROW_CHUNK, src.height - row))
            arr = src.read(window=win).reshape(src.count, -1).T.astype("float64")
            ok = ~(arr == NODATA).any(axis=1) & np.isfinite(arr).all(axis=1)
            res = np.full((arr.shape[0], NB), NODATA, dtype="float32")
            if ok.any():
                p = model.predict(arr[ok], quantiles=QUANTILES)
                di = aoa.di(arr[ok])
                res[ok, :5] = p
                res[ok, 5] = p[:, 4] - p[:, 0]
                res[ok, 6] = di
                res[ok, 7] = (di <= aoa.threshold).astype("float32")
            dst.write(res.T.reshape(NB, win.height, win.width), window=win)
            print(f"\r  predicting rows {row + win.height}/{src.height}", end="")
        dst.descriptions = tuple(BAND_NAMES)
    print(f"\n  -> {out_path}")

    # ---- quick-look map
    with rasterio.open(out_path) as r:
        b = {n: i + 1 for i, n in enumerate(BAND_NAMES)}
        med = np.ma.masked_equal(r.read(b["q50"]), NODATA)
        width = np.ma.masked_equal(r.read(b["pi90_width"]), NODATA)
        aoa_b = np.ma.masked_equal(r.read(b["aoa"]), NODATA)
        ext = [r.bounds.left, r.bounds.right, r.bounds.bottom, r.bounds.top]
    fig, axes = plt.subplots(1, 3, figsize=(17, 6), constrained_layout=True)
    panels = [(med, f"{label} - median prediction", "RdYlBu" if prop == "ph" else "YlOrBr"),
              (width, "90% prediction interval width", "magma_r"),
              (aoa_b, f"Area of applicability - white = inside (DI ≤ {aoa.threshold:.2f})", "Greys_r")]
    for ax, (img, title, cmap) in zip(axes, panels):
        im = ax.imshow(img, extent=ext, cmap=cmap)
        ax.scatter(d.x, d.y, s=2, c="k")
        ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
        if title.startswith("Area"):
            continue
        fig.colorbar(im, ax=ax, shrink=0.8)
    inside = float(aoa_b.mean())
    fig.suptitle(f"{label}, 0-30 cm   |   {inside:.0%} of the area inside the AOA")
    fig.savefig(OUT_MAPS / f"{prop}_0_30cm.png", dpi=130)
    plt.close(fig)

pd.DataFrame(summary).round(3).to_csv(OUT_M / "cv_summary.csv", index=False)
print("\nCV summary\n", pd.DataFrame(summary).round(3).to_string(index=False))


=== pH: n = 500
  spatial  ME=-0.025  RMSE=0.510  MAE=0.409  R2_MEC=0.281  CCC=0.460  PICP90=0.900  PICP50=0.498
  random   ME=0.009  RMSE=0.455  MAE=0.361  R2_MEC=0.429  CCC=0.605  PICP90=0.902  PICP50=0.530
  top covariates: lst_night, aridity_idx, tmean_x10, elev, rain_mm
  predicting rows 1321/1321
  -> outputs/maps/ph_0_30cm.tif

=== SOC (g/kg): n = 500
  spatial  ME=-1.124  RMSE=6.569  MAE=4.913  R2_MEC=0.282  CCC=0.465  PICP90=0.840  PICP50=0.510
  random   ME=0.137  RMSE=5.205  MAE=3.818  R2_MEC=0.549  CCC=0.708  PICP90=0.904  PICP50=0.542
  top covariates: elev, lst_day, lst_night, tmean_x10, rain_seas
  predicting rows 1321/1321
  -> outputs/maps/soc_gkg_0_30cm.tif

CV summary
 property      cv   n     ME  RMSE   MAE  R2_MEC   CCC  PICP90  PICP50
      ph spatial 500 -0.025 0.510 0.409   0.281 0.460   0.900   0.498
      ph  random 500  0.009 0.455 0.361   0.429 0.605   0.902   0.530
 soc_gkg spatial 500 -1.124 6.569 4.913   0.282 0.465   0.840   0.510
 soc_gkg  random 500  

STEP 4: Step 4 - Indicators: classify prediction rasters using config/indicator_thresholds.csv

For every property with a prediction raster (outputs/maps/<prop>_0_30cm.tif,
bands q05, q50, q95, pi90_width, di, aoa) this writes:

  outputs/indicators/<prop>_classes.tif   uint8, nodata 0, 5 bands:
      1 class_q50     class of the median prediction (the headline class)
      2 class_q05     class of the lower bound
      3 class_q95     class of the upper bound
      4 confidence    1 = high (q05..q95 in one class), 2 = medium (two adjacent
                      classes), 3 = low (three or more classes)
      5 aoa           1 = inside area of applicability, 2 = outside
  outputs/indicators/<prop>_class_summary.csv   area (km2, %) per class,
                                                all pixels and inside the AOA
  outputs/indicators/<prop>_classes.png         quick-look maps

Thresholds are read from the config, so changing classes or adding a property
needs no code change - only rows in the CSV (and METHOD below if relevant).
Run from the repo root:  python scripts/04_indicators.py

In [18]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

THRESHOLDS = "config/indicator_thresholds.csv"
MAPS = Path("outputs/maps")
OUT = Path("outputs/indicators")
DEPTH = "0_30cm"
NODATA_IN = -9999
# which method's thresholds to use per property (matches the 'method' column)
METHOD = {"ph": "H2O", "soc_gkg": "any"}
CMAPS = {"ph": "RdYlBu", "soc_gkg": "YlOrBr"}

OUT.mkdir(parents=True, exist_ok=True)
thr = pd.read_csv(THRESHOLDS)


def classify(values, table):
    """Class id (1..n) for each value; 0 where value is nodata."""
    lows = table["min"].fillna(-np.inf).to_numpy()
    ids = table["class_id"].to_numpy()
    idx = np.searchsorted(lows, values, side="right") - 1
    out = ids[np.clip(idx, 0, len(ids) - 1)].astype("uint8")
    out[~np.isfinite(values)] = 0
    return out


for prop in thr.property.unique():
    src_path = MAPS / f"{prop}_{DEPTH}.tif"
    if not src_path.exists():
        print(f"[skip] {prop}: no prediction raster at {src_path}")
        continue
    table = thr[(thr.property == prop) &
                (thr.method == METHOD.get(prop, "any"))].sort_values("class_id")
    if table.empty:
        print(f"[skip] {prop}: no thresholds for method {METHOD.get(prop)}")
        continue

    with rasterio.open(src_path) as src:
        band = {name: i + 1 for i, name in enumerate(src.descriptions)}
        q = src.read([band["q25"], band["q50"], band["q75"]]).astype("float64")
        aoa = src.read(band["aoa"])
        prof = src.profile
        px_km2 = abs(src.transform.a * src.transform.e) / 1e6
        extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    q[q == NODATA_IN] = np.nan

    c25, c50, c75 = (classify(q[i], table) for i in (0, 1, 2))
    valid = c50 > 0
    span = np.abs(c75.astype(int) - c25.astype(int))
    conf = np.where(span == 0, 1, np.where(span == 1, 2, 3)).astype("uint8")
    conf[~valid] = 0
    aoa_b = np.where(aoa == 1, 1, 2).astype("uint8")
    aoa_b[~valid] = 0

    # ---- raster
    prof.update(count=5, dtype="uint8", nodata=0, compress="deflate")
    out_tif = OUT / f"{prop}_classes.tif"
    with rasterio.open(out_tif, "w", **prof) as dst:
        dst.write(np.stack([c50, c25, c75, conf, aoa_b]))
        dst.descriptions = ("class_q50", "class_q25", "class_q75", "confidence", "aoa")

    # ---- area summary
    rows = []
    for _, r in table.iterrows():
        m = c50 == r.class_id
        rows.append({"class_id": r.class_id, "class": r["class"],
                     "range": (f"< {r['max']:g}" if pd.isna(r["min"]) else
                               f">= {r['min']:g}" if pd.isna(r["max"]) else
                               f"{r['min']:g} - <{r['max']:g}") + f" {r.unit}",
                     "area_km2": m.sum() * px_km2,
                     "area_km2_in_aoa": (m & (aoa_b == 1)).sum() * px_km2})
    summ = pd.DataFrame(rows)
    summ["pct"] = 100 * summ.area_km2 / summ.area_km2.sum()
    summ["pct_in_aoa"] = 100 * summ.area_km2_in_aoa / max(summ.area_km2_in_aoa.sum(), 1e-9)
    conf_pct = {k: 100 * (conf == v).sum() / valid.sum()
                for k, v in [("high", 1), ("medium", 2), ("low", 3)]}
    summ.round(1).to_csv(OUT / f"{prop}_class_summary.csv", index=False)

    print(f"\n=== {prop}  ->  {out_tif}")
    print(summ.round(1).to_string(index=False))
    print("confidence (% of area): " +
          ", ".join(f"{k} {v:.0f}%" for k, v in conf_pct.items()))

    # ---- quick-look
    n = len(table)
    colors = plt.get_cmap(CMAPS.get(prop, "viridis"))(np.linspace(0.05, 0.95, n))
    fig, axes = plt.subplots(1, 2, figsize=(14, 7), constrained_layout=True)
    img = np.ma.masked_equal(c50, 0)
    axes[0].imshow(img, cmap=ListedColormap(colors), vmin=0.5, vmax=n + 0.5,
                   extent=extent, interpolation="nearest")
    present = set(np.unique(c50[valid]))
    axes[0].legend(handles=[Patch(color=colors[i], label=f"{r['class']}")
                            for i, (_, r) in enumerate(table.iterrows())
                            if r.class_id in present],
                   loc="lower left", fontsize=8)
    axes[0].set_title(f"{prop} class (median prediction)")
    cc = ["#1a9850", "#fee08b", "#d73027"]
    axes[1].imshow(np.ma.masked_equal(conf, 0), cmap=ListedColormap(cc),
                   vmin=0.5, vmax=3.5, extent=extent, interpolation="nearest")
    axes[1].legend(handles=[Patch(color=c, label=l) for c, l in
                            zip(cc, ["High - one class", "Medium - two classes",
                                     "Low - three or more"])],
                   loc="lower left", fontsize=8)
    axes[1].set_title("Class confidence (50% interval, q25-q75)")
    for ax in axes:
        ax.set_xticks([]); ax.set_yticks([])
    fig.suptitle(f"{prop} indicator, {DEPTH.replace('_', '-')}  |  thresholds: "
                 f"{table.source.iloc[0].split(' - ')[0]}", fontsize=10)
    fig.savefig(OUT / f"{prop}_classes.png", dpi=130)
    plt.close(fig)


=== ph  ->  outputs/indicators/ph_classes.tif
 class_id              class               range  area_km2  area_km2_in_aoa  pct  pct_in_aoa
        1     Extremely acid      < 4.5 pH units      35.6             35.4  0.6         0.8
        2 Very strongly acid 4.5 - <5.1 pH units    1645.9           1604.0 29.7        34.7
        3      Strongly acid 5.1 - <5.6 pH units    2479.2           1858.7 44.8        40.2
        4    Moderately acid 5.6 - <6.1 pH units    1372.3           1120.0 24.8        24.2
        5      Slightly acid 6.1 - <6.6 pH units       1.4              1.4  0.0         0.0
        6            Neutral 6.6 - <7.4 pH units       0.0              0.0  0.0         0.0
        7  Slightly alkaline 7.4 - <7.9 pH units       0.0              0.0  0.0         0.0
        8           Alkaline     >= 7.9 pH units       0.0              0.0  0.0         0.0
confidence (% of area): high 2%, medium 77%, low 21%

=== soc_gkg  ->  outputs/indicators/soc_gkg_classes.tif
 class